# Parse instruments
Version the reference data carried by checked FIX market messages.


In [ ]:
project_root = "."
# Checked FIX market messages, as `parse_fix_market` wrote them.
source = "fix.market"
start = None
end = None
fix_dictionary = None
catalog = {"name": "rekep", "properties": {}}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target = "market.instruments"
batch_row_size = 65_536
commit_batch_num = 8
commit_row_size = None
log_level = "INFO"

In [ ]:
from pyiceberg.expressions import And, GreaterThanOrEqual, In, IsNull, LessThan

from rekep.fix.registry import FixRegistry
from rekep.iceberg import IcebergCatalog
from rekep.logs import Stage, configure
from rekep.market import InstUpdate
from rekep.text import FixMsg
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


# The FIX stage resolved the transaction clock and wrote it as `unix`, which is
# what this table is partitioned from. The read requests event order explicitly;
# pipeline writes do not add a physical sort.
lower, upper = unix_of(start), unix_of(end, upper=True)
if isinstance(batch_row_size, bool) or not isinstance(batch_row_size, int):
    raise TypeError("batch_row_size must be an integer")
if batch_row_size <= 0:
    raise ValueError("batch_row_size must be positive")
if isinstance(commit_batch_num, bool) or not isinstance(commit_batch_num, int):
    raise TypeError("commit_batch_num must be an integer")
if commit_batch_num <= 0:
    raise ValueError("commit_batch_num must be positive")
if commit_row_size is not None and (
    isinstance(commit_row_size, bool) or not isinstance(commit_row_size, int)
):
    raise TypeError("commit_row_size must be an integer or null")
if commit_row_size is not None and commit_row_size <= 0:
    raise ValueError("commit_row_size must be positive")
registry = (
    FixRegistry()
    if fix_dictionary is None
    else FixRegistry(cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root))
)
field = FixMsg.into_field()
store = IcebergCatalog.from_dict(catalog)
messages = store.dataset(
    source,
    field=field,
    branch=branch,
)
instruments = store.dataset(
    target,
    field=InstUpdate.into_field(),
    table_properties=dict(table_properties),
    branch=branch,
    commit_batch_num=commit_batch_num,
    commit_row_size=commit_row_size,
)
stage = Stage(
    "parse_instruments",
    sources={"market": source},
    targets={"instruments": target},
    window=(lower, upper),
)

In [ ]:
read = written = 0


def _observed():
    """Every update the window's messages describe, enriched per ticker."""
    if not messages.exists:
        return iter(())
    window = _window(lower, upper)
    # Failed rows remain in fix.market for audit; an incomplete reading cannot
    # become reference data merely because its raw message carried a symbol.
    clean = IsNull("error")
    row_filter = clean if window is None else And(window, clean)
    reader = messages.read_arrow_reader(
        field, row_filter=row_filter, order_by=("unix", "msgseqnum", "hash")
    )
    return InstUpdate.from_fixmsgs(FixMsg.from_arrow_reader(reader), registry=registry)


def _stored(codes):
    """Current rows keyed by their canonical component ticker."""
    if not instruments.exists or not codes:
        return {}
    reader = instruments.read_arrow_reader(
        InstUpdate.into_field(), row_filter=In("code", codes)
    )
    return {
        row.instrument.symbolticker: row
        for row in InstUpdate.from_arrow_reader(reader)
    }


def _versions():
    """Only what changes the table, one bounded lookup per batch."""
    global read, written
    for batch in InstUpdate.into_arrow_reader(_observed(), batch_row_size=batch_row_size):
        observed = list(InstUpdate.from_arrow_reader(iter((batch,))))
        read += len(observed)
        stored = _stored(tuple(row.code for row in observed))
        changed = list(InstUpdate.versioned(observed, stored))
        if changed:
            written += len(changed)
            yield InstUpdate.into_arrow_batch(changed)


def _with_first(first, rest):
    yield first
    yield from rest


# One lifecycle has one current row under the declared xhash primary key, so
# enrichment overwrites it. Nothing is written when no batch changed, so a
# replay of an unchanged window commits no snapshot.
changes = iter(_versions())
first = next(changes, None)
if first is not None:
    instruments.overwrite_arrow_reader(
        _with_first(first, changes),
        InstUpdate.into_field(),
        merge_by=True,
        commit_row_size=commit_row_size,
        commit_batch_num=commit_batch_num,
    )

stage.says("observed %d instruments, of which %d are new versions", read, written)
result = stage.finished(read=read, written=written)
result